# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 3:** government offices (*Comune*, residence, ID card, tax code) · driver's licence · bank & bills · train and air travel · dining out · graduation (*maturità*, *laurea*) · Italian holidays

Grammar in context: formal imperative (*compili, firmi, prenda*), impersonal *si*, *bisogna*, future, passive with *essere*, *ci vuole / ci vogliono*, reflexive passato prossimo (*mi sono trasferito, mi sono laureato*). Part 8 reviews sentences from Lessons 1–2.

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson03_VitaQuotidiana_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 3 (B1)
# Theme: government offices, driver's licence, banking & bills,
#        travel, dining out, graduation, Italian holidays
# Grammar: formal imperative (prenda, compili), impersonal si,
#          bisogna + infinitive / bisogna che + subjunctive, future,
#          passive with essere, ci vuole / ci vogliono, reflexive past
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson three.
(narrator) Today you will deal with real life in Italy. Government offices, the bank, bills, trains, restaurants, graduation, and holidays.
(narrator) As always, answer out loud in the pause. Then repeat after the Italian speaker.
(narrator) Listen to this conversation. Marco has moved to a new city. He is at the town hall, {il Comune}, to get an identity card.
(pause 1)
(it F) Buongiorno. Ha preso il numero?
(it M) Sì, ho il numero quarantasette. Ho aspettato quasi un'ora!
(it F) Mi dispiace, oggi c'è molta gente. Come posso aiutarla?
(it M) Vorrei fare la carta d'identità. Mi sono trasferito qui il mese scorso.
(it F) Ha già chiesto la residenza?
(it M) Sì, ecco il certificato di residenza e il codice fiscale.
(it F) Perfetto. Mi serve anche una fototessera.
(it M) Eccola. Quanto costa?
(it F) Ventidue euro. Si paga con il bancomat o con la carta.
(it M) Un'ultima domanda. Per convertire la patente straniera, dove devo andare?
(it F) Deve andare alla Motorizzazione. Ma bisogna prendere un appuntamento online.
(it M) Va bene. E quando sarà pronta la carta d'identità?
(it F) Arriverà a casa Sua entro sei giorni lavorativi.
(it M) Grazie mille. È stata molto gentile.
(pause 1.5)

(section) Part 1 — At the town hall
(narrator) Let's look at this conversation, piece by piece.
(narrator) In many Italian offices, you take a number and wait. To take a number is {prendere il numero}.
(repeat x2 slow) Ha preso il numero?
(prompt) Ask, formally: did you take a number? || Ha preso il numero?
(prompt) Answer: yes, I have number forty-seven. || Sì, ho il numero quarantasette.
(prompt) Say: I waited almost an hour! || Ho aspettato quasi un'ora!
(narrator) To wait in line is {fare la fila}.
(prompt) Say: I waited in line for two hours. || Ho fatto la fila per due ore.
(prompt) Say: today there are a lot of people. || Oggi c'è molta gente.
(narrator) The identity card is {la carta d'identità}.
(build) tità | identità | carta d'identità
(prompt) Say: I would like to get an identity card. Use {fare}. || Vorrei fare la carta d'identità.
(narrator) To move, to a new city, is {trasferirsi}. In the past: {mi sono trasferito}. A woman says {mi sono trasferita}.
(build) ferirsi | trasferirsi
(repeat x2 slow) Mi sono trasferito qui il mese scorso.
(prompt) Say: I moved here last month. || Mi sono trasferito qui il mese scorso.
(prompt) Ask, formally: did you take a number? || Ha preso il numero?

(narrator) Your official address is {la residenza}. The tax code is {il codice fiscale}. Every person in Italy needs one.
(repeat x2 slow) il codice fiscale
(prompt) Ask, formally: have you already applied for residence? || Ha già chiesto la residenza?
(prompt) Say: here is the residence certificate and the tax code. || Ecco il certificato di residenza e il codice fiscale.
(narrator) A passport photo is {una fototessera}.
(build) tessera | fototessera
(prompt) The clerk says: I also need a passport photo. || Mi serve anche una fototessera.
(prompt) Say: here it is. Talk about the photo. || Eccola.
(narrator) Now the impersonal {si}. {Si paga} means: one pays, or you pay.
(prompt) Say: you pay by debit card or by credit card. || Si paga con il bancomat o con la carta.
(prompt) Ask: where does one pay? || Dove si paga?
(prompt) Say: I would like to get an identity card. || Vorrei fare la carta d'identità.
(prompt) Say: I moved here last month. || Mi sono trasferito qui il mese scorso.

(narrator) Clerks often give orders with the formal imperative. Listen. Fill in this form.
(repeat x2 slow) Compili questo modulo.
(prompt) Say, formally: fill in this form. || Compili questo modulo.
(prompt) Say, formally: sign here, please. || Firmi qui, per favore.
(prompt) Say, formally: take a number and wait. || Prenda il numero e aspetti.
(prompt) Say, formally: come back tomorrow morning. || Torni domani mattina.
(narrator) An appointment is {un appuntamento}. {Bisogna} means: it is necessary, one must.
(prompt) Say: you need to make an appointment online. || Bisogna prendere un appuntamento online.
(prompt) Ask: when will the identity card be ready? || Quando sarà pronta la carta d'identità?
(prompt) Answer: it will arrive at your home within six working days. Formal. || Arriverà a casa Sua entro sei giorni lavorativi.
(prompt) Say: I waited in line for two hours. || Ho fatto la fila per due ore.
(prompt) Say: here is the tax code. || Ecco il codice fiscale.

(section) Part 2 — The driver's licence
(narrator) The driver's licence is {la patente}. The office for vehicles and licences is {la Motorizzazione}.
(build) zione | rizzazione | Motorizzazione
(prompt) Ask: to convert a foreign driver's licence, where do I have to go? || Per convertire la patente straniera, dove devo andare?
(prompt) Answer, formally: you have to go to the Motorizzazione. || Deve andare alla Motorizzazione.
(narrator) Some foreign licences can be converted. For others, you must take the exams.
(narrator) A driving school is {una scuola guida}. There is a theory exam, {l'esame di teoria}, and a driving test, {l'esame di guida}.
(prompt) Say: I signed up at a driving school. || Mi sono iscritto a una scuola guida.
(prompt) Say: I passed the theory exam. || Ho superato l'esame di teoria.
(prompt) Say: the driving test is next week. || L'esame di guida è la settimana prossima.
(narrator) To fail an exam, Italians often say {essere bocciato}.
(prompt) Say: I failed the first time. || Sono stato bocciato la prima volta.
(prompt) Say: I got my driver's licence at eighteen. || Ho preso la patente a diciotto anni.
(prompt) Say: my licence expires next year. || La mia patente scade l'anno prossimo.
(prompt) Say: you need to make an appointment online. || Bisogna prendere un appuntamento online.
(prompt) Say, formally: fill in this form. || Compili questo modulo.

(section) Part 3 — Bank and bills
(narrator) Now the bank. A bank account is {un conto corrente}.
(build) rente | corrente | conto corrente
(prompt) Say: I would like to open a bank account. || Vorrei aprire un conto corrente.
(prompt) Ask: what documents do I need? || Quali documenti mi servono?
(prompt) Answer, formally: you need an identity document and the tax code. || Le servono un documento d'identità e il codice fiscale.
(narrator) The cash machine, and the debit card, are both called {il bancomat}. To withdraw money is {prelevare}.
(prompt) Say: I need to withdraw money. || Devo prelevare dei soldi.
(prompt) Say: the cash machine is out of order. || Il bancomat non funziona.
(narrator) A bank transfer is {un bonifico}. Your account number in Europe is the {IBAN}.
(prompt) Say: I made a bank transfer. || Ho fatto un bonifico.
(prompt) Ask, formally: can you give me your IBAN? || Mi può dare il suo IBAN?
(prompt) Say: I would like to open a bank account. || Vorrei aprire un conto corrente.

(narrator) Bills are {le bollette}. Electricity is {la luce}. Gas is {il gas}.
(build) lette | bollette | le bollette
(prompt) Say: I have to pay the bills. || Devo pagare le bollette.
(prompt) Say: the electricity bill is due on the twentieth. || La bolletta della luce scade il venti.
(prompt) Say: this month the gas bill is very high. || Questo mese la bolletta del gas è molto alta.
(prompt) Say: I paid the bill online. || Ho pagato la bolletta online.
(prompt) Say: you can also pay at the post office. Use the impersonal {si}. || Si può pagare anche alla posta.
(narrator) Automatic payment from your account is {l'addebito diretto}.
(prompt) Say: I would like automatic payment on my account. || Vorrei l'addebito diretto sul conto.
(prompt) Say: I got my driver's licence at eighteen. || Ho preso la patente a diciotto anni.
(prompt) Say: I made a bank transfer. || Ho fatto un bonifico.

(section) Part 4 — Travelling
(narrator) Now let's travel by train. The platform is {il binario}. A return ticket is {un biglietto di andata e ritorno}.
(repeat x2 slow) un biglietto di andata e ritorno
(prompt) Say: a return ticket to Florence, please. || Un biglietto di andata e ritorno per Firenze, per favore.
(prompt) Ask: what time does the next train for Florence leave? || A che ora parte il prossimo treno per Firenze?
(prompt) Ask: which platform does it leave from? || Da quale binario parte?
(prompt) Answer: from platform nine. || Dal binario nove.
(narrator) On regional trains, you must validate a paper ticket. To validate is {convalidare}.
(build) dare | validare | convalidare
(prompt) Say: remember to validate the ticket! Speak informally. || Ricordati di convalidare il biglietto!
(prompt) Say: the train is twenty minutes late. || Il treno è in ritardo di venti minuti.
(narrator) A strike is {uno sciopero}. In Italy, strikes are announced in advance.
(prompt) Say: tomorrow there is a train strike. || Domani c'è lo sciopero dei treni.
(prompt) Say: I missed the connection. || Ho perso la coincidenza.
(prompt) Say: the electricity bill is due on the twentieth. || La bolletta della luce scade il venti.

(narrator) Now the airport. The boarding pass is {la carta d'imbarco}. Hand luggage is {il bagaglio a mano}.
(prompt) Say: I printed the boarding pass. || Ho stampato la carta d'imbarco.
(prompt) Say: I only have hand luggage. || Ho solo il bagaglio a mano.
(narrator) For the passive voice, use {essere} plus the past participle.
(prompt) Say: the flight was cancelled. || Il volo è stato cancellato.
(prompt) Say: my suitcase was lost. || La mia valigia è stata persa.
(narrator) To say how long a trip takes, use {ci vuole} for one thing, and {ci vogliono} for many.
(prompt) Ask: how long does it take from Rome to Milan? || Quanto ci vuole da Roma a Milano?
(prompt) Answer: by high-speed train, it takes three hours. || Con il treno veloce ci vogliono tre ore.
(prompt) Ask: which platform does it leave from? || Da quale binario parte?
(prompt) Say: the train is twenty minutes late. || Il treno è in ritardo di venti minuti.

(section) Part 5 — Dining out
(narrator) Now let's eat. An Italian meal has a first course, {il primo}, usually pasta. Then a second course, {il secondo}, meat or fish.
(prompt) Ask, formally: what do you recommend? Use {consigliare}, and "to us". || Che cosa ci consiglia?
(prompt) Say: for the first course, I'll have the carbonara. || Per primo prendo la carbonara.
(prompt) Say: and for the second course, the grilled fish. || E per secondo il pesce alla griglia.
(prompt) Say: a bottle of still water, please. || Una bottiglia d'acqua naturale, per favore.
(prompt) Say: I am allergic to nuts. || Sono allergico alla frutta secca.
(prompt) Ask: is there gluten in this dish? || C'è glutine in questo piatto?
(narrator) To ask for the bill, say {il conto}. Many restaurants add a cover charge: {il coperto}.
(prompt) Say: the bill, please. || Il conto, per favore.
(prompt) Ask: is the cover charge included? || Il coperto è incluso?
(prompt) Ask: can we pay separately? || Possiamo pagare separatamente?
(narrator) In Italy, a tip, {la mancia}, is not required. But it is welcome.
(prompt) Say: in Italy, the tip is not required. || In Italia la mancia non è obbligatoria.
(prompt) Say: I would like to open a bank account. || Vorrei aprire un conto corrente.
(prompt) Say: tomorrow there is a train strike. || Domani c'è lo sciopero dei treni.

(narrator) A role-play. You are at a restaurant. Answer the waiter.
(it M) Buonasera. Avete deciso?
(prompt) Say: not yet. What do you recommend? || Non ancora. Che cosa ci consiglia?
(it M) Oggi abbiamo un ottimo risotto ai funghi.
(prompt) Say: perfect. For the first course, I'll have the risotto. || Perfetto. Per primo prendo il risotto.
(it M) E da bere?
(prompt) Say: a bottle of still water, please. || Una bottiglia d'acqua naturale, per favore.
(it M) Benissimo. Altro?
(prompt) Say: is there gluten in this dish? I am allergic. || C'è glutine in questo piatto? Sono allergico.
(it M) No, il risotto è senza glutine.
(prompt) Later, say: the bill, please. Can we pay separately? || Il conto, per favore. Possiamo pagare separatamente?

(section) Part 6 — Graduation
(narrator) Now graduation. At the end of high school, Italian students take a big exam. It is called {l'esame di maturità}.
(build) rità | maturità | l'esame di maturità
(prompt) Say: I took the final high school exam five years ago. || Ho fatto la maturità cinque anni fa.
(prompt) Say: I graduated from a science high school. Use {diplomarsi}. || Mi sono diplomato al liceo scientifico.
(narrator) At university, you present your thesis to a committee. This is {la discussione della tesi}. After it, they announce your grade. This is {la proclamazione}.
(build) zione | mazione | proclamazione
(prompt) Say: I presented my thesis yesterday. || Ieri ho discusso la tesi.
(prompt) Say: I graduated with one hundred ten with honours. || Mi sono laureato con centodieci e lode.
(narrator) Italian university grades go up to one hundred and ten. New graduates wear a laurel crown: {la corona d'alloro}. And anyone with a degree can be called {dottore}, or {dottoressa}.
(prompt) Say to a new graduate: congratulations, doctor! Speak to a man. || Congratulazioni, dottore!
(prompt) Say: after graduation we threw a big party. || Dopo la laurea abbiamo fatto una grande festa.
(prompt) Say, to a friend: I am proud of you. || Sono orgoglioso di te.
(prompt) Say: I would like to do a master's abroad. || Vorrei fare un master all'estero.
(prompt) Ask, formally: what do you recommend? || Che cosa ci consiglia?
(prompt) Say: the flight was cancelled. || Il volo è stato cancellato.

(section) Part 7 — Italian holidays
(narrator) Finally, Italian holidays, {le feste}. Christmas is {Natale}. New Year's Day is {Capodanno}.
(prompt) Say: at Christmas, we stay with the family. || A Natale stiamo con la famiglia.
(narrator) On the sixth of January, {l'Epifania}, an old woman called {la Befana} brings sweets to children.
(prompt) Say: the Befana brings sweets to children. || La Befana porta i dolci ai bambini.
(narrator) Easter is {Pasqua}. The Monday after Easter is {Pasquetta}. Many Italians go on a picnic. There is an old proverb.
(repeat x2 slow) Natale con i tuoi, Pasqua con chi vuoi.
(narrator) It means: Christmas with your family, Easter with whoever you want.
(prompt) Say: on Easter Monday, we go on a picnic in the countryside. || A Pasquetta facciamo un picnic in campagna.
(narrator) Two national holidays: the twenty-fifth of April, {la Festa della Liberazione}, and the second of June, {la Festa della Repubblica}.
(prompt) Say: on the second of June, we celebrate the Republic. || Il due giugno si festeggia la Repubblica.
(narrator) The fifteenth of August is {Ferragosto}. It is the heart of the summer holidays.
(build) gosto | Ferragosto
(prompt) Say: at Ferragosto, many shops are closed. || A Ferragosto molti negozi sono chiusi.
(prompt) Say: everybody goes to the sea. || Tutti vanno al mare.
(narrator) When a holiday falls on a Thursday, many people also take Friday off. This is called {fare il ponte}: making the bridge.
(prompt) Say: the twenty-fifth of April falls on a Thursday. || Il venticinque aprile cade di giovedì.
(prompt) Say: so we'll take the long weekend. Use {fare il ponte}. || Quindi faremo il ponte.
(narrator) Some greetings. Happy holidays is {buone feste}. Happy New Year is {buon anno}.
(prompt) Say: happy holidays to everyone! || Buone feste a tutti!
(prompt) Say: Merry Christmas and happy New Year! || Buon Natale e felice anno nuovo!
(prompt) Say: I graduated with one hundred ten with honours. || Mi sono laureato con centodieci e lode.
(prompt) Say: I have to pay the bills. || Devo pagare le bollette.

(section) Part 8 — Review
(narrator) Let's review. First, sentences from lessons one and two.
(prompt) Say: I would like to register for your course. || Vorrei iscrivermi al suo corso.
(prompt) Say: I would like to do my thesis on photonics. || Vorrei fare la tesi sulla fotonica.
(prompt) Ask: do you have it in a bigger size? || Ce l'ha in una taglia più grande?
(prompt) Say: I would like two hundred grams of it. || Ne vorrei due etti.
(prompt) Say: the package hasn't arrived yet. || Il pacco non è ancora arrivato.
(prompt) Say: I would like to book a table for four people, for Saturday evening. || Vorrei prenotare un tavolo per quattro persone, per sabato sera.
(narrator) Now today's lesson. Answer quickly.
(prompt) Say: I waited in line for two hours. || Ho fatto la fila per due ore.
(prompt) Say: I moved here last month. || Mi sono trasferito qui il mese scorso.
(prompt) Say: you need to make an appointment online. || Bisogna prendere un appuntamento online.
(prompt) Say: I passed the theory exam. || Ho superato l'esame di teoria.
(prompt) Say: the electricity bill is due on the twentieth. || La bolletta della luce scade il venti.
(prompt) Ask: what time does the next train for Florence leave? || A che ora parte il prossimo treno per Firenze?
(prompt) Say: I missed the connection. || Ho perso la coincidenza.
(prompt) Ask: can we pay separately? || Possiamo pagare separatamente?
(prompt) Say to a new graduate: congratulations, doctor! || Congratulazioni, dottore!
(prompt) Say: at Ferragosto, many shops are closed. || A Ferragosto molti negozi sono chiusi.

(narrator) Now you are Marco, at the town hall. Answer the clerk.
(it F) Buongiorno. Come posso aiutarla?
(prompt M) Say: I would like to get an identity card. I moved here last month. || Vorrei fare la carta d'identità. Mi sono trasferito qui il mese scorso.
(it F) Ha il codice fiscale?
(prompt M) Say: yes, here is the residence certificate and the tax code. || Sì, ecco il certificato di residenza e il codice fiscale.
(it F) Benissimo. Si paga con il bancomat o con la carta.
(prompt M) Ask: when will the identity card be ready? || Quando sarà pronta la carta d'identità?
(it F) Arriverà a casa Sua entro sei giorni lavorativi.
(prompt M) Say: thank you very much. You have been very kind. || Grazie mille. È stata molto gentile.

(narrator) Now listen to the first conversation again.
(pause 1)
(it F) Buongiorno. Ha preso il numero?
(it M) Sì, ho il numero quarantasette. Ho aspettato quasi un'ora!
(it F) Mi dispiace, oggi c'è molta gente. Come posso aiutarla?
(it M) Vorrei fare la carta d'identità. Mi sono trasferito qui il mese scorso.
(it F) Ha già chiesto la residenza?
(it M) Sì, ecco il certificato di residenza e il codice fiscale.
(it F) Perfetto. Mi serve anche una fototessera.
(it M) Eccola. Quanto costa?
(it F) Ventidue euro. Si paga con il bancomat o con la carta.
(it M) Un'ultima domanda. Per convertire la patente straniera, dove devo andare?
(it F) Deve andare alla Motorizzazione. Ma bisogna prendere un appuntamento online.
(it M) Va bene. E quando sarà pronta la carta d'identità?
(it F) Arriverà a casa Sua entro sei giorni lavorativi.
(it M) Grazie mille. È stata molto gentile.
(pause 1.5)
(narrator) This is the end of lesson three. Listen to it again tomorrow. {Buone feste, e buon viaggio!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
